In [ ]:
import pydicom
import dicom2nifti
from pathlib import Path

# Paths
dicom_study_dir = Path("")
fixed_dir = dicom_study_dir.parent / "T0_fixed"
output_nifti_dir = dicom_study_dir.parent / "T0_Nifti_Out"

fixed_dir.mkdir(exist_ok=True)
output_nifti_dir.mkdir(exist_ok=True)

print("Injecting the official DICOM signature into files...")
# Fix files
for f in dicom_study_dir.glob("*"):
    if f.is_file():
        ds = pydicom.dcmread(f, force=True)

        # --- 1. THE FILE_META FIX ---
        # Create the "envelope" if it doesn't exist
        if not hasattr(ds, 'file_meta'):
            ds.file_meta = pydicom.dataset.FileMetaDataset()
            
        # Retrieve the TransferSyntaxUID from the file body (DICOM tag is 0x0002, 0x0010)
        if (0x0002, 0x0010) in ds:
            ds.file_meta.TransferSyntaxUID = ds[0x0002, 0x0010].value
        elif not hasattr(ds.file_meta, 'TransferSyntaxUID'):
            # Desperate edge case
            ds.file_meta.TransferSyntaxUID = pydicom.uid.ImplicitVRLittleEndian

        ds.decompress()

        # THE MAGIC TRICK: Create 128 empty bytes. pydicom will automatically add "DICM" after these!
        ds.preamble = b"\0" * 128 
        ds.save_as(fixed_dir / f.name)

Iniettando la firma ufficiale DICOM nei file...


In [ ]:
def inspect_dicom_signature(file_path):
    print(f"🔍 Inspecting: {Path(file_path).name}")
    try:
        # Open the file in 'rb' mode (read binary)
        with open(file_path, 'rb') as f:
            # Read the first 132 bytes (128 empty + 4 signature)
            first_132_bytes = f.read(132)
            
            # Extract exactly bytes 128 to 131
            signature = first_132_bytes[128:132]
            
            print(f"  File length read: {len(first_132_bytes)} bytes")
            
            if signature == b'DICM':
                print(f"SIGNATURE FOUND: {signature}")
            else:
                # If not present, print what's really in those bytes
                print(f"SIGNATURE MISSING! Found this: {signature}")
                
            # Bonus: Print the first 10 absolute bytes of the file for curiosity
            print(f"First 10 bytes: {first_132_bytes[:10]}\n")            
            print(f"Bytes 128 to 131: {first_132_bytes[128:132].hex()}\n")
            
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
# INSERT YOUR PATHS HERE
# 1. A file from the original dataset (e.g., from T0 folder)
broken_file = r""

# 2. A file we have repaired (from T0_fixed folder)
repaired_file = r""

print("--- TEST ORIGINAL FILE ---")
inspect_dicom_signature(broken_file)

print("--- TEST REPAIRED FILE ---")
inspect_dicom_signature(repaired_file)

--- TEST FILE ORIGINALE ---
🔍 Ispezionando: NCT07132658_5_MR1xT_image00000.dcm
  Lunghezza file letto: 132 bytes
FIRMA MANCANTE! Ho trovato questo: b'840.'
Primi 10 byte: b'\x08\x00\x05\x00\n\x00\x00\x00IS'

Byte da 128 a 131: 3834302e

--- TEST FILE RIPARATO ---
🔍 Ispezionando: NCT07132658_5_MR1xT_image00000.dcm
  Lunghezza file letto: 132 bytes
FIRMA TROVATA: b'DICM'
Primi 10 byte: b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'

Byte da 128 a 131: 4449434d



In [ ]:
# CONVERSION TO NIfTI
print("Trying conversion to NIfTI on corrected files...")
try:
    dicom2nifti.convert_directory(
        str(dicom_study_dir), 
        str(output_nifti_dir), 
        compression=True, 
        reorient=True
    )
    print(f"✅ Conversion completed! Check the folder: {output_nifti_dir}")
except Exception as e:
    print(f"❌ Error: {e}")

Provo la conversione in NIfTI sui file corretti...
✅ Conversione completata! Controlla la cartella: C:\Users\dosimetria4\Naiara_gdr\data\prove\anonymized_Dataset\NCT07132658_5\T0_Nifti_Out
